# Chapter 16 — Keeping Your Priorities Straight with Heaps

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 16

## The problem: priority queues
A **priority queue** is like a queue (you only access/remove from the **front**), but insertion is like an **ordered array**: every new item goes to its place by **priority**, not arrival time.

Example: an **ER triage** system. Patients are treated by severity (1–10), not by who arrived first. Someone with severity 10 jumps ahead of the flu patient who's been waiting for hours.

Like stacks and queues (Ch. 9), a priority queue is an **abstract data type**: it describes *behaviour*, and you choose the structure underneath.

| Underlying structure | Insert | Delete (take highest priority) |
|---|---|---|
| Ordered array (highest priority at the **end**) | **O(N)** | O(1) |
| **Heap** | **O(log N)** | **O(log N)** |

## 1. What is a heap?

We use the **binary max-heap**: a binary tree with two rules.

1. **Heap condition:** every node is **greater than all of its descendants**.
   (A **min-heap** flips it: every node is *smaller* than its descendants. Everything else is identical.)
2. **Complete tree:** every level is completely filled, **except possibly the bottom row**, and the bottom row fills **from the left** with no gaps before the last node.

Example max-heap:

                 100
              /       \
            88         25
          /    \      /  \
        87      16   8    12
       /  \    /  \  /
      86  50  2  15 3        <- "last node" = 3 (rightmost on the bottom row)

### Heap vs BST: don't mix them up
| | BST | Heap |
|---|---|---|
| Rule | left < node < right | node > **all** descendants (left and right are not ordered among themselves) |
| Search | O(log N) | useless: you can't tell whether to go left or right → O(N) |
| Ordering | strict | **weakly ordered** |
| Special power | sorted order | **the root is always the max** |

**Last node:** the rightmost node on the bottom level. It's central to both operations.

## 2. Insertion: "trickle up", O(log N)

1. Put the new value at the **next free spot** on the bottom row (it becomes the new last node). This keeps the tree complete.
2. Compare it with its **parent**. If it's **bigger**, swap them.
3. Repeat until its parent is bigger, or it reaches the root.

Example: insert 40 → becomes the child of 8 → 40 > 8 swap → 40 > 25 swap → 40 < 100 stop.
A node moves up at most log N levels → **O(log N)**.

## 3. Deletion: "trickle down", O(log N)

You only ever delete the **root** (the highest priority, the front of the priority queue).

1. Move the **last node** into the root's place (the old root is gone).
2. "Trickle node" = the new root. Look at its two children and pick the **larger** one.
3. If the trickle node is smaller than that larger child, **swap** them.
4. Repeat until no child is bigger.

**Why the larger child?** If you swap with the smaller child, that smaller value becomes the parent of the larger one, which immediately breaks the heap condition.

Example: delete 100 → 3 goes to the root → swap with 88 → with 87 → with 86 → done.

### Why a heap beats an ordered array for priority queues
| | Ordered array | Heap |
|---|---|---|
| Insert | O(N), **slow** | O(log N), very fast |
| Delete | O(1), extremely fast | O(log N), very fast |

Priority queues do inserts and deletes in roughly **equal amounts** (everyone who enters the ER gets treated). *Consistently very fast* beats *sometimes extremely fast, sometimes slow.*

## 4. The Problem of the Last Node → solution: store the heap in an array

Why insist on using the last node? Inserting or deleting anywhere else would make the tree **incomplete**, so the tree becomes **imbalanced**, and imbalanced trees lose O(log N). (Same lesson as the BST with sorted inserts.)

But with linked nodes, *finding* the last node would mean checking every node.

**The twist:** implement the heap as a **plain array**. Number the nodes level by level, left to right:

    index:   0    1   2   3   4  5   6   7   8  9  10  11
    value: [100, 88, 25, 87, 16, 8, 12, 86, 50, 2, 15, 3]

- Root = `data[0]`
- **Last node = `data[-1]`** → found in O(1). Insert = `append`, delete = `pop`.

Navigation formulas (for the node at index `i`):

| To find… | Formula |
|---|---|
| left child | `2*i + 1` |
| right child | `2*i + 2` |
| parent | `(i - 1) // 2` (integer division) |

Check: index 4 (value 16) → children at 9 and 10 (2 and 15), parent at 1 (88). ✓

This is the first time the book uses an **array to store a tree**. Any binary tree can be stored this way, but for heaps it's what makes them efficient.

In [ ]:
class Heap:
    """Binary MAX-heap stored in a Python list."""

    def __init__(self):
        self.data = []

    def root_node(self):
        return self.data[0] if self.data else None

    def last_node(self):
        return self.data[-1] if self.data else None

    # --- navigation formulas ---
    def left_child_index(self, i):
        return 2 * i + 1

    def right_child_index(self, i):
        return 2 * i + 2

    def parent_index(self, i):
        return (i - 1) // 2


def show_heap(data):
    """Print the heap level by level."""
    level, start = 0, 0
    while start < len(data):
        size = 2 ** level
        print(f"level {level}: {data[start:start + size]}")
        start += size
        level += 1


h = Heap()
h.data = [100, 88, 25, 87, 16, 8, 12, 86, 50, 2, 15, 3]   # the book's example heap
show_heap(h.data)
print(h.left_child_index(4), h.right_child_index(4), h.parent_index(4))   # 9 10 1

In [ ]:
def insert(self, value):
    self.data.append(value)                       # 1) new value becomes the last node
    i = len(self.data) - 1

    # 2) trickle up: while not the root AND bigger than its parent
    while i > 0 and self.data[i] > self.data[self.parent_index(i)]:
        p = self.parent_index(i)
        self.data[p], self.data[i] = self.data[i], self.data[p]   # swap with parent
        i = p                                     # continue from the new position

Heap.insert = insert

h = Heap()
h.data = [100, 88, 25, 87, 16, 8, 12, 86, 50, 2, 15, 3]
h.insert(40)
show_heap(h.data)        # 40 has risen to level 1, under 100

In [ ]:
def has_greater_child(self, i):
    l, r, n = self.left_child_index(i), self.right_child_index(i), len(self.data)
    return (l < n and self.data[l] > self.data[i]) or \
           (r < n and self.data[r] > self.data[i])

def calculate_larger_child_index(self, i):
    l, r = self.left_child_index(i), self.right_child_index(i)
    if r >= len(self.data):                       # no right child
        return l
    return r if self.data[r] > self.data[l] else l

def delete(self):
    """Remove and RETURN the root (the maximum)."""
    if not self.data:
        return None
    root = self.data[0]
    last = self.data.pop()                        # take the last node off the end
    if self.data:                                 # (if the heap isn't empty now)
        self.data[0] = last                       # 1) last node replaces the root
        i = 0
        while self.has_greater_child(i):          # 2) trickle down
            larger = self.calculate_larger_child_index(i)
            self.data[i], self.data[larger] = self.data[larger], self.data[i]
            i = larger
    return root

Heap.has_greater_child = has_greater_child
Heap.calculate_larger_child_index = calculate_larger_child_index
Heap.delete = delete

h = Heap()
h.data = [100, 88, 25, 87, 16, 8, 12, 86, 50, 2, 15, 3]
print(h.delete())        # 100
show_heap(h.data)        # 88 is the new root; the 3 trickled down to where 86 was

> **Python differences from the book's Ruby (on purpose):**
> - Ruby returns `nil` for an out-of-range index; Python raises `IndexError`. So the child checks compare the index with `len(self.data)`.
> - Ruby treats `0` as truthy and Python doesn't. Checking indices instead of values avoids a bug when the heap contains `0`.
> - The book's `@data[0] = @data.pop` crashes in Python on a 1-element heap, because the list is empty after `pop`. My `delete` handles that **and returns the removed root**, which is what a priority queue needs.

## 5. Heaps as priority queues, and Python's `heapq`

- Highest priority = the root → read it in **O(1)**.
- Remove it → **O(log N)**, and the next-highest floats to the top.
- Add a new item → **O(log N)**.

The heap's **weak ordering** is its strength: it doesn't fully sort, so inserts are cheap, but it's ordered *just enough* to always know the max.

**In real Python code you use `heapq`.** It works on a plain list, and you've seen why that works.
⚠️ `heapq` is a **min-heap** (smallest first). For a max-heap, push **negated** priorities.
Push `(priority, item)` tuples: tuples compare element by element, so the first element decides.

Link to Operations Research: priority queues sit inside **Dijkstra's shortest path**, **event-driven simulation** (next event = smallest timestamp) and **scheduling** (e.g. shortest-processing-time rules). It's a building block you'll use again.

## Summary
- Priority queue = abstract data type; the **heap** is its best foundation.
- Max-heap: node > all descendants + **complete** tree → root = maximum.
- Insert: add as last node, **trickle up**. Delete root: last node to the root, **trickle down** (swap with the larger child). Both **O(log N)**.
- Stored as an **array**: last node = `data[-1]`; children `2i+1`, `2i+2`; parent `(i-1)//2`.
- Heaps are bad for search (O(N)), and that's fine: you only ever need the top.

In [ ]:
import heapq

# ER triage: (severity, patient). heapq is a MIN-heap -> negate severity for "highest first"
er = []
for severity, patient in [(3, "E"), (10, "C"), (6, "A"), (1, "B")]:
    heapq.heappush(er, (-severity, patient))

while er:
    neg_sev, patient = heapq.heappop(er)
    print(f"treat patient {patient} (severity {-neg_sev})")

## Exercises (book solutions: p. 458)

### The heap for exercises 1 and 2
Drawn as a tree (and as an array, using the level-by-level numbering):

                 10
              /      \
             9        8
           /   \     / \
          6     5   7   4
         / \   /
        2   1 3

    array: [10, 9, 8, 6, 5, 7, 4, 2, 1, 3]

### Exercise 1: insert
**Input:** the heap above.
**Task:** on paper, **insert 11**. Where does it start (next free spot on the bottom row → whose child?), and which swaps happen as it trickles up? Draw the final heap. Then check with the code below.

### Exercise 2: delete the root
**Input:** the **original** heap above (*not* the one after exercise 1).
**Task:** on paper, **delete the root (10)**. Which node moves to the top, and which swaps happen as it trickles down? Remember: always swap with the **larger** child. Draw the result, then check.

### Exercise 3: heap as a sorter
**Input:** an empty max-heap. Insert these numbers **in this order**: `55, 22, 34, 10, 2, 99, 68`.
**Task:** then **delete (pop) the root** 7 times, appending each removed value to a new list. **Predict the final list** before running the code. (Hint: think about what the root always is. This idea is the basis of an algorithm called *heapsort*.)

In [ ]:
# Exercise 1 — draw first, then check:
h1 = Heap()
h1.data = [10, 9, 8, 6, 5, 7, 4, 2, 1, 3]
# h1.insert(11)
# show_heap(h1.data)

# Exercise 2 — draw first, then check (fresh copy of the ORIGINAL heap):
h2 = Heap()
h2.data = [10, 9, 8, 6, 5, 7, 4, 2, 1, 3]
# h2.delete()
# show_heap(h2.data)

# Exercise 3 — predict first, then check:
h3 = Heap()
for x in [55, 22, 34, 10, 2, 99, 68]:
    h3.insert(x)
# result = [h3.delete() for _ in range(7)]
# print(result)